# **06 - Explainability Generation Node**

## **1. Pipeline Objective**

## **2. Dependency Management**

### **2.1 Package Installation**

In [1]:
!pip install -q transformers accelerate bitsandbytes datasets pyarrow pandas

### **2.2 Module Imports**

In [2]:
import datetime
import json
import os
import time
import re
import gc

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

### **2.3 Environment Variables**

In [3]:
PROJECT_ROOT: str = '/content/drive/MyDrive/LLMReliability'
DATA_PROCESSED_DIR: str = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODELS_DIR: str = os.path.join(PROJECT_ROOT, 'models')
RESULTS_DIR: str = os.path.join(PROJECT_ROOT, 'results')

MODEL_ID: str = "Qwen/Qwen3-4B-Instruct-2507"
OUT_PATH: str = os.path.join(DATA_PROCESSED_DIR, 'explainability_subset.jsonl')

os.environ['HF_HOME'] = MODELS_DIR

for d in [DATA_PROCESSED_DIR, MODELS_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

## **3. Resource Allocation**

### **3.1 Identity & Access Management (IAM)**

In [4]:
try:
    from google.colab import drive
    drive.mount('/content/drive')

    secrets_path = '/content/drive/MyDrive/secrets.json'
    if os.path.exists(secrets_path):
        with open(secrets_path, 'r') as f:
            secrets = json.load(f)
        hf_token = secrets.get("HF_TOKEN")
        if hf_token:
            from huggingface_hub import login
            login(token=hf_token)
            print("[ENVIRONMENT] status=AUTHENTICATED | component='HuggingFace'")
except ImportError:
    print("[ENVIRONMENT] status=LOCAL | component='Google Drive'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[ENVIRONMENT] status=AUTHENTICATED | component='HuggingFace'


### **3.2 Dataset Loading (Subset)**

In [5]:
os.chdir(PROJECT_ROOT)

df = pd.read_parquet('data/processed/indopref_main.parquet')
subset_df = df.head(5).copy()
print(
    f"[DATASET] shape={df.shape} | "
    f"subset={len(subset_df)} | "
    f"status=LOADED"
)

[DATASET] shape=(369, 10) | subset=5 | status=LOADED


## **4. Model Loading**

In [6]:
print(f"[MODEL] id='{MODEL_ID}' | status=INITIALIZING")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=MODELS_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    cache_dir=MODELS_DIR
)
print(f"[MODEL] status=LOADED | vram='{torch.cuda.memory_allocated() / 1024**3:.2f} GB'")

[MODEL] id='Qwen/Qwen3-4B-Instruct-2507' | status=INITIALIZING


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[MODEL] status=LOADED | vram='2.49 GB'


## **5. Prompt Engineering**

In [7]:
def format_json_prompt(user_input: str, resp_A: str, resp_B: str) -> str:
    return (
        "### PERAN\n"
        "Anda adalah Juri AI (LLM-as-a-Judge) tingkat lanjut yang sangat objektif dan analitis.\n\n"
        "### KONTEKS\n"
        "Anda harus mengevaluasi dua respons AI berdasarkan kriteria berikut:\n"
        "1. Accuracy: Apakah respons memuat informasi yang faktual dan benar?\n"
        "2. Relevance: Apakah respons menjawab instruksi secara langsung?\n"
        "3. Clarity: Apakah respons terstruktur dengan baik dan tidak bertele-tele?\n\n"
        "### INSTRUKSI\n" + user_input + "\n\n"
        "### RESPONS A\n" + resp_A + "\n\n"
        "### RESPONS B\n" + resp_B + "\n\n"
        "### AKSI\n"
        "1. Ekstrak bukti spesifik (evidence) dari kedua respons untuk setiap kriteria.\n"
        "2. Berikan skor 1-10 untuk masing-masing kriteria.\n"
        "3. Tuliskan penjelasan analitis (Chain-of-Thought).\n"
        "4. Tentukan pemenang akhir (A atau B).\n\n"
        "### FORMAT JAWABAN\n"
        "Keluarkan hasil evaluasi Anda HANYA dalam format JSON valid tanpa awalan/akhiran markdown backticks (```json ... ```).\n"
        "PENTING: JSON harus dieksekusi secara terurut. Atribut 'explanation' HARUS berada di atas sebelum 'decision'.\n"
        "Gunakan skema persis seperti berikut:\n"
        "{\n"
        "  \"explanation\": {\n"
        "    \"accuracy\": \"Penjelasan singkat\",\n"
        "    \"relevance\": \"Penjelasan singkat\",\n"
        "    \"clarity\": \"Penjelasan singkat\"\n"
        "  },\n"
        "  \"evidence\": [\n"
        "    {\n"
        "      \"criterion\": \"accuracy/relevance/clarity\",\n"
        "      \"candidate\": \"A atau B\",\n"
        "      \"evidence\": \"kutipan teks respons\",\n"
        "      \"assessment\": \"supported atau unsupported\"\n"
        "    }\n"
        "  ],\n"
        "  \"criteria\": {\n"
        "    \"accuracy\": { \"A\": 9, \"B\": 8 },\n"
        "    \"relevance\": { \"A\": 8, \"B\": 9 },\n"
        "    \"clarity\": { \"A\": 9, \"B\": 9 }\n"
        "  },\n"
        "  \"decision\": \"A\"\n"
        "}"
    )


## **6. Explainability Compute Engine**

### **6.1 Inference Function Definition**

In [8]:
def run_inference(user_input: str, resp_A: str, resp_B: str) -> str:
    raw_prompt = format_json_prompt(user_input, resp_A, resp_B)
    messages = [
        {"role": "system", "content": "You are an impartial and expert AI evaluator. Output valid raw JSON without markdown formatting."},
        {"role": "user", "content": raw_prompt}
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_len = inputs['input_ids'].shape[1]

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=2048,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][input_len:]
    res = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    del inputs, outputs, generated_tokens
    torch.cuda.empty_cache()
    gc.collect()

    return res

### **6.2 Compute Loop**

In [9]:
class InferenceTelemetry:
    def __init__(self, total_steps, log_interval=1):
        self.total_steps = total_steps
        self.log_interval = log_interval
        self.current_step = 0
        self.start_time = time.time()

    def update(self, step_size=1, **kwargs):
        self.current_step += step_size
        if self.current_step % self.log_interval == 0 or self.current_step == self.total_steps:
            now = time.time()
            elapsed = now - self.start_time
            avg_time = elapsed / self.current_step if self.current_step > 0 else 0
            eta_seconds = (self.total_steps - self.current_step) * avg_time

            eta_str = str(datetime.timedelta(seconds=int(eta_seconds)))
            pct = (self.current_step / self.total_steps) * 100

            metrics = " | ".join([f"{k}: {v}" for k, v in kwargs.items()])
            metrics_str = f" | {metrics}" if metrics else ""

            print(f"[TELEMETRY] Step: {self.current_step}/{self.total_steps} ({pct:.1f}%) | ETA: {eta_str}{metrics_str}")

In [10]:
results = []
print(
    f"[INFERENCE] samples={len(subset_df)} | "
    f"mode='symmetrical' | "
    f"status=STARTED"
)

telemetry = InferenceTelemetry(total_steps=len(subset_df), log_interval=1)
for index, row in subset_df.iterrows():
    raw_ab = run_inference(row['input'], row['response_1'], row['response_2'])

    raw_ba = run_inference(row['input'], row['response_2'], row['response_1'])

    telemetry.update(
        1, 
        ID=row['original_id'], 
        CharsAB=len(raw_ab), 
        CharsBA=len(raw_ba)
    )

    results.append({
        "sample_id": row["original_id"],
        "prompt": row["input"],
        "cand_1": row["response_1"],
        "cand_2": row["response_2"],
        "human_preference": row.get("chosen_model", "Unknown"),
        "json_output_ab": raw_ab,
        "json_output_ba": raw_ba
    })

with open(OUT_PATH, 'w', encoding='utf-8') as f:
    for res in results:
        f.write(json.dumps(res, ensure_ascii=False) + '\n')

print(f"[INFERENCE] status=COMPLETED | path='{OUT_PATH}'")

[INFERENCE] samples=5 | mode='symmetrical' | status=STARTED
[TELEMETRY] Step: 1/5 (20.0%) | ETA: 0:08:58 | ID: 238 | CharsAB: 2629 | CharsBA: 2640
[TELEMETRY] Step: 2/5 (40.0%) | ETA: 0:08:25 | ID: 327 | CharsAB: 3339 | CharsBA: 3170
[TELEMETRY] Step: 3/5 (60.0%) | ETA: 0:06:07 | ID: 423 | CharsAB: 3236 | CharsBA: 4135
[TELEMETRY] Step: 4/5 (80.0%) | ETA: 0:02:59 | ID: 403 | CharsAB: 2551 | CharsBA: 3816
[TELEMETRY] Step: 5/5 (100.0%) | ETA: 0:00:00 | ID: 454 | CharsAB: 2851 | CharsBA: 3985
[INFERENCE] status=COMPLETED | path='/content/drive/MyDrive/LLMReliability/data/processed/explainability_subset.jsonl'
